In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

#tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]

In [0]:
%sql

select * from vcredit_medallion.bronze.clientes where data_cadastro like '%-%'

In [0]:
%sql

-- verificar se a idade não é formada apenas por dígitos
select * from vcredit_medallion.bronze.clientes where idade NOT REGEXP '^[0-9]+$';



In [0]:
def normalizar_texto(col):
    r"""
    Normaliza texto:
    - tudo minúsculo
    - mantém só letras (com acento) e números
    - todo o resto (pontuação, símbolos, '_', \n, \t, \r e outros espaços) vira espaço
    - sequências de espaço viram um só, e trim nas pontas
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    return F.trim(
        F.regexp_replace(F.lower(c.cast("string")), r"[^\p{L}\p{M}0-9]+", " ")
    )

In [0]:
def normalizar_inteiro(col):
    """
    Remove tudo que não for dígito (0-9) e converte para inteiro (bigint).
    - texto sem nenhum dígito (ex.: 'NULL', '', 'abc') vira null
    - mais de 18 dígitos vira null (evita estouro do bigint)
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    digitos = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return (
        F.when((digitos == "") | (F.length(digitos) > 18), F.lit(None).cast("long"))
         .otherwise(digitos.cast("long"))
    )

In [0]:
def mostrar_padrao_data(df, datetime_column):
    # Substitui qualquer dígito numérico (0-9) pelo número 9
    # Assim, "25/10/2023" vira "99/99/9999" e "2023-05-12" vira "9999-99-99"
    df_padroes = df.withColumn(
        "padrao_data",
        F.regexp_replace(F.col(datetime_column), r"\d", "9")
    )

    # Agrupa e conta a frequência de cada padrão encontrado no seu dataset
    df_padroes.groupBy("padrao_data").count().orderBy(F.desc("count")).show()

In [0]:
df_bronze_clientes = spark.table(f"{bronze_schema}.clientes")

In [0]:
# Verifica padrões de data na coluna data_cadastro
mostrar_padrao_data(df_bronze_clientes, "data_cadastro")

# Padrões identificados: dd/MM/yyyy e yyyy-MM-dd

In [0]:
df_silver_clientes = (df_bronze_clientes
                      .dropDuplicates(["id_cliente"])
                      .withColumn("nome", normalizar_texto("nome"))
                      .withColumn("regiao", normalizar_texto("regiao"))
                      .withColumn("email", F.trim("email"))
                      .withColumn("idade", normalizar_texto("idade"))
                      .withColumn("dia_vencimento_fatura", normalizar_texto("dia_vencimento_fatura"))
                      .withColumn("data_cadastro",
                    F.coalesce(
                    F.try_to_timestamp("data_cadastro", F.lit("yyyy-MM-dd")),
                    F.try_to_timestamp("data_cadastro", F.lit("dd/MM/yyyy"))
                        )))
                      

mostrar_padrao_data(df_silver_clientes, "data_cadastro")

display(df_silver_clientes.limit(30))